In [14]:
import sys
!{sys.executable} -m pip install requests pandas plotly streamlit --quiet
print


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\bitap\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [15]:
import requests
import pandas as pd
import plotly.express as px
import time
from datetime import datetime, timedelta, timezone

print("All libraries imported successfully!")
print(f' pandas version: {pd.__version__}')
print(f' plotly version: {requests.__version__}')


All libraries imported successfully!
 pandas version: 2.1.3
 plotly version: 2.32.3


In [16]:
BASE_URL = 'https://pam-stilling-feed.nav.no'

def get_public_token():
    print('Fetching public token from NAV...')
    response = requests.get(f'{BASE_URL}/api/publicToken') 
    
    if response.status_code == 200: # check if the request was successful
        token = response.text.strip().strip('"') # remove any surrounding quotes
        print(f'Token received! First 30 characters: {token[:30]}...') # print the first 30 characters of the token for verification
        return token
    else:
        print(f'Something went wrong. Status code: {response.status_code}')
        return None
    
TOKEN = get_public_token()
TOKEN = TOKEN.split('\n', 1)[-1].strip()
    

Fetching public token from NAV...
Token received! First 30 characters: Current public token for Nav J...


In [17]:
def fetch_jobs(token, max_pages=15):
    all_jobs = []
    
    thirty_days_ago = datetime.now(timezone.utc) - timedelta(days=30)
    date_str = thirty_days_ago.strftime('%a, %d %b %Y %H:%M:%S GMT') # This is the format expected by the API for the 'publishedSince' parameter which is used to filter jobs published in the last 30 days. %a is the abbreviated weekday name, %d is the day of the month, %b is the abbreviated month name, %Y is the year with century, %H is the hour (24-hour clock), %M is the minute, %S is the second, and GMT indicates that the time is in Greenwich Mean Time.
    print(f'Fetching jobs published since: {date_str}')
    
    headers = {
        'Accept': 'application/json', # Accept header tells the server that we want the response in JSON format, which is easier to work with in Python.
        'Authorization': f'Bearer {token}', # Authorization header is used to authenticate our request using the token we obtained earlier. The 'Bearer' prefix indicates that we are using a Bearer token for authentication. A Bearer token is a type of access token that is typically used in OAuth 2.0 authentication to grant access to protected resources. By including this header, we are proving to the server that we have the necessary permissions to access the job data.
        'If-Modified-Since': date_str
    } # This header tells the server to only return jobs that have been modified (published) since the specified date, which helps us filter for jobs published in the last 30 days.
    
    url = f'{BASE_URL}/api/v1/feed' # This is the endpoint for fetching job listings from the NAV API. The '/api/v1/feed' path indicates that we want to access the version 1 of the feed API, which provides job listings in a structured format. By sending a GET request to this URL with the appropriate headers, we can retrieve the job data that we need for our dashboard.
    
    
    for page_num in range(1, max_pages + 1):
        print(f'Fetching page {page_num}/{max_pages}...', end=' ')
        
        response = requests.get(url, headers=headers)
        
        if response.status_code == 304:
            print('No more new jobs')
            break
        elif response.status_code != 200:
            print(f'Error fetching jobs: {response.status_code}')
            break
        
        data = response.json() # Parse the JSON response into a Python dictionary. This allows us to easily access the job data and convert it into a DataFrame for analysis and visualization.
        items = data.get('items', []) # Extract the list of job items from the response. The 'items' key contains the actual job listings, and we use the .get() method to safely access it, providing an empty list as a default value in case the key is not present in the response.
        print(f'Got {len(items)} jobs.') 
        
        
        # Print the raw structure of the first item
        # import json
        # print(json.dumps(items[0], indent=2))
        
        for item in items:
            entry = item.get('_feed_entry', {})
            job = {
                'title'     : entry.get('title', 'Unknown'),
                'employer'  : entry.get('businessName', 'Unknown'),
                'location'  : entry.get('municipal', 'Unknown'),
                'status'    : entry.get('status', 'Unknown'),
                'last_updated': entry.get('sistEndret', '')
            }
            
            
            all_jobs.append(job)
            
            
        next_url = data.get('next_url')
        if not next_url:
            print('Reached the end of the feed.')
            break
            
        url = BASE_URL + next_url # Update the URL for the next page of results. The 'next_url' provided in the response is a relative path, so we prepend the BASE_URL to construct the full URL for the next request.
        time.sleep(0.5) # Sleep for a short time to avoid overwhelming the server with requests. This is a good practice when making multiple requests in a loop, as it helps to prevent hitting rate limits and allows the server to handle the load more effectively.
            
            
    print(f'Done! Total jobs collected: {len(all_jobs)}')
    return all_jobs
        
jobs_list = fetch_jobs(TOKEN, max_pages=15)
            

Fetching jobs published since: Sat, 09 May 2026 12:00:16 GMT
Fetching page 1/15... Got 1000 jobs.
Fetching page 2/15... Got 1000 jobs.
Fetching page 3/15... Got 1000 jobs.
Fetching page 4/15... Got 1000 jobs.
Fetching page 5/15... Got 1000 jobs.
Fetching page 6/15... Got 1000 jobs.
Fetching page 7/15... Got 1000 jobs.
Fetching page 8/15... Got 1000 jobs.
Fetching page 9/15... Got 1000 jobs.
Fetching page 10/15... Got 1000 jobs.
Fetching page 11/15... Got 1000 jobs.
Fetching page 12/15... Got 1000 jobs.
Fetching page 13/15... Got 1000 jobs.
Fetching page 14/15... Got 1000 jobs.
Fetching page 15/15... Got 1000 jobs.
Done! Total jobs collected: 15000


In [18]:
df = pd.DataFrame(jobs_list)
df = df[df['status'] == 'ACTIVE']
df = df[df['title'] != '...']
df = df.reset_index(drop=True)

print(f'shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()

print('First 5 rows: ')
df.head()

shape: 10528 rows x 5 columns

First 5 rows: 


,title,employer,location,status,last_updated
0,Sommerjobb som resepsjonist,Adecco Norge AS,BODØ,ACTIVE,2026-05-09T14:12:16.030816+02:00
1,Vi søker sesonghjelp over 18 til REMA 1000 Bre...,REMA 1000 Breivika,ÅLESUND,ACTIVE,2026-05-09T14:20:47.522089+02:00
2,Åpen søknad for REMA 1000 Breivika,REMA 1000 Breivika,ÅLESUND,ACTIVE,2026-05-09T14:22:17.609803+02:00
3,Barnehagemedarbeider,Capella,BERGEN,ACTIVE,2026-05-09T15:32:18.846657+02:00
4,Dyktig og engasjert fagarbeider/assistent søkes!,Sandefjord kommune Gokstad skole,SANDEFJORD,ACTIVE,2026-05-09T15:38:49.630319+02:00


In [19]:
df_active = df[df['status'] == 'ACTIVE'].copy()

print(f'Total jobs fetched: {len(df)}')
print(f'Active jobs: {len(df_active)}')
print() # This code filters the DataFrame to include only rows where the 'status' column is 'ACTIVE', and then prints the total number of jobs fetched and the number of active jobs. The .copy() method is used to create a new DataFrame for active jobs, which helps to avoid potential issues with chained indexing when modifying the filtered DataFrame later on.


df_active['location'] = df_active['location'].str.title() # This code converts the 'location' column in the df_active DataFrame to title case, which means that the first letter of each word will be capitalized. This is done to improve the readability and consistency of the location names when we visualize the data later on. For example, "oslo" would become "Oslo", and "bergen" would become "Bergen".
df_active['last_updated'] = pd.to_datetime(df_active['last_updated'], errors='coerce') # This code converts the 'last_updated' column in the df_active DataFrame to datetime objects. The pd.to_datetime() function is used for this conversion, and the errors='coerce' parameter ensures that any values that cannot be parsed as dates will be set to NaT (Not a Time) instead of raising an error. This allows us to work with the 'last_updated' column as actual datetime objects, which can be useful for sorting, filtering, and visualizing the data based on the last updated time of the job listings.

print('Sample of cleaned data:')
df_active[['title', 'employer', 'location', 'last_updated']].head(10)



Total jobs fetched: 10528
Active jobs: 10528

Sample of cleaned data:


,title,employer,location,last_updated
0,Sommerjobb som resepsjonist,Adecco Norge AS,Bodø,2026-05-09 14:12:16.030816+02:00
1,Vi søker sesonghjelp over 18 til REMA 1000 Bre...,REMA 1000 Breivika,Ålesund,2026-05-09 14:20:47.522089+02:00
2,Åpen søknad for REMA 1000 Breivika,REMA 1000 Breivika,Ålesund,2026-05-09 14:22:17.609803+02:00
3,Barnehagemedarbeider,Capella,Bergen,2026-05-09 15:32:18.846657+02:00
4,Dyktig og engasjert fagarbeider/assistent søkes!,Sandefjord kommune Gokstad skole,Sandefjord,2026-05-09 15:38:49.630319+02:00
5,Alfa Hårstudio søker ny frisør.,Alfa Hårstudio Rr As,Stord,2026-05-09 16:09:24.357531+02:00
6,Sales Agent / Distribution Partner wanted for ...,Reshape Healthcare,?,2026-05-09 16:09:44.391515+02:00
7,Alfa Hårstudio søker ny frisør.,Alfa Hårstudio Rr As,Stord,2026-05-09 16:10:44.417896+02:00
8,Vi søker deltidsmedarbeidere!,Esso Amtmannsvingen Deli de Luca,Lier,2026-05-09 16:13:45.074605+02:00
9,Vil du bli med å skape møbelbransjens beste ku...,Skeidar Tiller,Trondheim,2026-05-09 16:47:10.505335+02:00


In [20]:
CATEGORIES = {
    'Software / Developer'      : ['developer', 'software', 'programmer', 'frontend',
                                    'backend', 'fullstack', 'utvikler'],
    'Data / AI / ML'            : ['data', 'machine learning', 'scientist', 'analyst',
                                    'analytics', 'mlops', 'kunstig intelligens', 'ai '],
    'Engineering'               : ['engineer', 'ingeniør', 'tekniker', 'engineering'],
    'Teacher / Education'       : ['teacher', 'lærer', 'pedagog', 'professor', 'lektor'],
    'Consultant'                : ['consultant', 'konsulent', 'advisor', 'rådgiver'],
    'Research / Science'        : ['researcher', 'research', 'forsker', 'forskning', 'phd'],
    'Color / Optics / Imaging'  : ['color', 'colour', 'farge', 'colorimetry', 'colorimetric',
                                    'spectral', 'spectroscopy', 'spektral', 'spektroskopi',
                                    'optical', 'optikk', 'imaging', 'appearance', 'photometry',
                                    'chromatic', 'illumination', 'camera', 'kamera', 'vision',
                                    'brdf', 'reflectance'],
    'Coating / Materials'       : ['coating', 'belegg', 'lakkering', 'paint', 'maling',
                                    'pigment', 'lacquer', 'varnish', 'primer', 'polymer',
                                    'surface treatment', 'overflatebehandling', 'material',
                                    'materialvitenskap', 'kompositt', 'composite'],
    'Chemistry / Lab'           : ['chemistry', 'kjemi', 'kjemiker', 'kjemisk', 'chemical',
                                    'laborator', 'lab ', 'synthesis', 'syntese', 'analytical',
                                    'analytisk', 'biokjemi', 'formulation', 'formulering'],
    'Quality / Inspection'      : ['quality', 'kvalitet', 'inspection', 'inspeksjon',
                                    'kontroll', 'qa ', 'qc ', 'testing', 'measurement',
                                    'måling', 'metrology', 'metrologi', 'standardization',
                                    'sertifisering', 'certification'],
    'Corrosion / Protection'    : ['corrosion', 'korrosjon', 'rust', 'anticorrosive',
                                    'protection', 'beskyttelse', 'degradation', 'cathodic',
                                    'inhibitor', 'passivation'],
    'Textile / Fiber'           : ['textile', 'tekstil', 'fiber', 'fabric', 'garment',
                                    'clothing', 'stoff', 'dyeing', 'farging', 'nonwoven',
                                    'yarn', 'garn'],
    'Other'                     : []
}

def categorise_job(title):
    title_lower = str(title).lower()
    for category, keywords in CATEGORIES.items():
        if category == 'Other':
            continue
        if any(kw in title_lower for kw in keywords):
            return category
    return 'Other'
    
    
df_active['category'] = df_active['title'].apply(categorise_job) # This code applies the categorise_job function to the 'title' column of the df_active DataFrame and creates a new column called 'category' to store the resulting job categories. The categorise_job function checks if any of the keywords defined in the CATEGORIES dictionary are present in the job title (converted to lowercase for case-insensitive matching) and assigns the corresponding category. If no keywords match, it assigns the category 'Other'. This allows us to group and analyze the job listings based on their categories later on in our dashboard.

print('Jobs per category: ')
print(df_active['category'].value_counts().to_string()) # This code prints the count of jobs in each category by using the value_counts() method on the 'category' column of the df_active DataFrame. The to_string() method is used to format the output as a string, which allows us to see the counts for all categories without truncation. This gives us an overview of how many job listings fall into each category, which can be useful for understanding the distribution of job types in our dataset.
    
    
    
    
    
    

Jobs per category: 
category
Other                     8473
Teacher / Education        937
Consultant                 512
Engineering                241
Data / AI / ML              96
Software / Developer        93
Research / Science          88
Quality / Inspection        58
Coating / Materials         18
Chemistry / Lab              7
Corrosion / Protection       4
Textile / Fiber              1


In [21]:
city_counts = df_active['location'].value_counts().head(15)

fig_cities = px.bar(
    x = city_counts.values,
    y = city_counts.index,
    title = 'Top 15 cities',
    labels = {'x': 'Number of jobs', 'y': 'City'},
    color = city_counts.values,
    color_continuous_scale = 'Blues', # This parameter sets the color scale for the bars in the bar chart. 'Blues' is a predefined color scale in Plotly that ranges from light blue to dark blue. By using this color scale, we can visually differentiate the bars based on their values, with higher values appearing in darker shades of blue and lower values in lighter shades. This helps to enhance the visual appeal of the chart and makes it easier to interpret the data at a glance.
    text = city_counts.values
)

fig_cities.update_layout(
    yaxis = {'categoryorder': 'total ascending'}, # This parameter updates the layout of the bar chart to order the y-axis categories (cities) based on the total count of jobs in ascending order. By setting 'categoryorder' to 'total ascending', the cities with fewer job listings will appear at the top of the chart, while those with more job listings will appear towards the bottom. This ordering can help to highlight smaller cities with job opportunities and provide a clearer visual comparison between the different locations.
    showlegend = False,
    height = 600
)

fig_cities.show()

In [22]:
cat_counts = df_active['category'].value_counts()

fig_cat = px.pie(
    values = cat_counts.values,
    names = cat_counts.index,
    title = 'Job Categories Distribution',
    hole = 0.4, # This parameter creates a donut chart by specifying the size of the hole in the center of the pie chart. A value of 0.4 means that the hole will be 40% of the radius of the pie chart, which can make the chart more visually appealing and easier to read by providing a clear separation between the segments.
    color_discrete_sequence=px.colors.qualitative.Set2 # This parameter sets the color sequence for the segments of the pie chart. 'Set2' is a predefined qualitative color palette in Plotly that consists of a variety of distinct colors. By using this color sequence, we can ensure that each category in the pie chart is represented by a different color, making it easier to differentiate between the categories and enhancing the overall visual appeal of the chart.
    
)

fig_cat.update_traces(textinfo='percent+label') # This parameter updates the traces of the pie chart to specify what information should be displayed on the chart. By setting 'textinfo' to 'percent', we are telling Plotly to display the percentage value of each segment directly on the pie chart. This allows viewers to quickly understand the proportion of each category in relation to the whole dataset without needing to refer to a legend or hover over the segments for details.
fig_cat.update_layout(height=500)
fig_cat.show()


In [23]:
employer_count = df_active['employer'].value_counts().head(15)

fig_emp = px.bar(
    x = employer_count.values,
    y = employer_count.index,
    title = 'Top 15 employers',
    labels = {'x': 'Number of jobs', 'y': 'Employer'},
    color = employer_count.values,
    color_continuous_scale = 'Greens',
    text= employer_count.values
)

fig_emp.update_layout(
    yaxis = {'categoryorder': 'total ascending'},
    showlegend = False,
    height = 600
)

fig_emp.show()

In [24]:
df_active['date'] = df_active['last_updated'].dt.date # This code creates a new column called 'date' in the df_active DataFrame by extracting just the date part from the 'last_updated' column, which contains datetime objects. The .dt accessor allows us to access the datetime properties of the 'last_updated' column, and .date extracts only the date (year, month, day) while ignoring the time component. This new 'date' column can be useful for analyzing trends in job postings over time, such as counting the number of jobs posted on each date or visualizing the distribution of job postings across different dates.

daily_counts = (
    df_active.groupby('date')
    .size()
    .reset_index(name='job_count')
    .sort_values('date')
)

fig_time = px.line(
    daily_counts,
    x = 'date',
    y = 'job_count',
    title = 'Number of active jobs posted over time',
    labels = {'date': 'Date', 'job_count': 'Number of active jobs'},
)

fig_time.update_traces(line_color='royalblue', marker_color='royalblue')
fig_time.update_layout(height=500)
fig_time.show()





In [25]:
print('=' * 50)
print('Summary of Norwegian job market')
print('-' * 50)
print(f'Active job listing: {len(df_active)}')
print(f'Unique employers: {df_active["employer"].nunique()}')
print(f'Locations: {df_active["location"].nunique()}')
print()
print('Top 5 cities:')
for city, count in df_active['location'].value_counts().head(5).items():
    print(f' {city:<20} {count} jobs')
print()
print('Top 5 categories:')
for category, count in  df_active['category'].value_counts().head(5).items():
    print(f'{category:<25} {count} jobs')
print()

Summary of Norwegian job market
--------------------------------------------------
Active job listing: 10528
Unique employers: 3214
Locations: 312

Top 5 cities:
 Oslo                 1714 jobs
 Bergen               607 jobs
 Trondheim            470 jobs
 Stavanger            360 jobs
 Tromsø               318 jobs

Top 5 categories:
Other                     8473 jobs
Teacher / Education       937 jobs
Consultant                512 jobs
Engineering               241 jobs
Data / AI / ML            96 jobs



In [26]:
output_path = 'norway_jobs_summary.csv'

df_active.to_csv(output_path, index=False)

print(f'Data saved to: {output_path}')


Data saved to: norway_jobs_summary.csv
